# Ultra-short standalone pseudobulk DE pipeline

No project imports: this notebook directly aggregates pseudobulk counts, runs PyDESeq2, and computes optional Reactome/GMT ORA and GSEA outputs.


In [46]:
from pathlib import Path

H5AD_PATH = "/Users/bastien.herve/Downloads/RRMAP2_xenium_all_samples.cellcharter.companion.ready.with_metadata.rerun.with_AnnoL1Curated_with_Region_Anno2to4Updated.h5ad"
GROUPBY = "Anno_L1_curated"
REPLICATE = "meta_sample_id"
SOURCE = "OPC"
REFERENCE = "Schwann cell"

COUNTS_LAYER = "counts"
MIN_CELLS = 20
MIN_CELL_COUNTS = 100
MIN_GENE_COUNTS = 100
MIN_REPLICATES = 3
MIN_PCT_EXPRESSED = 0.1
P_ADJUST_METHOD = "fdr_bh"
PADJ_CUTOFF = 0.05
LOG2FC_CUTOFF = 2
DESEQ2_FIT_TYPE = "mean"
N_CPUS = 4

PATHWAY_GMT = None
PATHWAY_ORGANISM = "Mouse"
PATHWAY_TOP_N = 20
PATHWAY_MIN_OVERLAP = 3
PATHWAY_MAX_SIZE = 500
PATHWAY_GSEA_PERMUTATIONS = 100
PATHWAY_GSEA_SEED = 0

stem = Path(H5AD_PATH).with_suffix("")
OUT_PREFIX = stem.parent / f"{stem.name}.{GROUPBY}.{SOURCE}_vs_{REFERENCE}"

In [28]:
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests


def dense_integer_counts(x):
    x = x.toarray() if sp.issparse(x) else np.asarray(x)
    x = np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    return np.rint(np.clip(x, 0, None)).astype(int)


def row_sum(x):
    return (
        np.asarray(x.sum(axis=1)).ravel()
        if sp.issparse(x)
        else np.asarray(x).sum(axis=1)
    )


def padjust(pvalues, method):
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full(pvalues.shape, np.nan, dtype=float)
    ok = np.isfinite(pvalues)
    method = str(method or "fdr_bh").replace("-", "_").lower()
    out[ok] = (
        pvalues[ok]
        if method in {"none", "raw"}
        else multipletests(np.clip(pvalues[ok], 0, 1), method=method)[1]
    )
    return out


def pct_positive(x, mask, cols):
    sub = x[np.asarray(mask, dtype=bool)][:, list(cols)]
    return (
        np.asarray((sub > 0).mean(axis=0)).ravel()
        if sp.issparse(sub)
        else np.mean(np.asarray(sub) > 0, axis=0)
    )


def clean_gene_set(genes):
    seen, cleaned = set(), []
    for gene in genes:
        gene = str(gene).strip()
        key = gene.lower()
        if gene and key not in seen:
            seen.add(key)
            cleaned.append(gene)
    return cleaned


def read_gmt(paths):
    paths = [paths] if isinstance(paths, (str, Path)) else list(paths)
    gene_sets = {}
    for path in paths:
        with Path(path).expanduser().open(encoding="utf-8", errors="replace") as handle:
            for line in handle:
                term, _, *genes = line.rstrip("\n\r").split("\t")
                if term and genes:
                    gene_sets[term] = clean_gene_set(genes)
    return gene_sets


def ora(genes, gene_sets, universe):
    selected = {str(g).lower() for g in genes}
    universe = {str(g).lower() for g in universe}
    rows = []
    for term, members in gene_sets.items():
        pathway = {str(g).lower() for g in members} & universe
        overlap = pathway & selected
        if len(overlap) >= PATHWAY_MIN_OVERLAP and len(pathway) <= PATHWAY_MAX_SIZE:
            pvalue = hypergeom.sf(
                len(overlap) - 1, len(universe), len(pathway), len(selected)
            )
            rows.append(
                {
                    "Term": term,
                    "Count": len(overlap),
                    "GeneRatio": len(overlap) / max(len(selected), 1),
                    "P-value": pvalue,
                }
            )
    table = pd.DataFrame(rows)
    if len(table):
        table["Adjusted P-value"] = padjust(table["P-value"], "fdr_bh")
        table["-log10 adjusted P-value"] = -np.log10(
            np.clip(table["Adjusted P-value"], np.nextafter(0, 1), 1)
        )
        table = table.sort_values(
            ["GeneRatio", "Adjusted P-value"], ascending=[False, True]
        ).head(PATHWAY_TOP_N)
    return table

In [29]:
adata = ad.read_h5ad(H5AD_PATH)
counts = adata.layers[COUNTS_LAYER] if COUNTS_LAYER else adata.X
group = adata.obs[GROUPBY].astype("category")
replicate = adata.obs[REPLICATE].astype(str)

In [30]:
valid = (group.cat.codes.to_numpy() >= 0) & replicate.notna().to_numpy()
valid &= row_sum(counts) >= int(MIN_CELL_COUNTS)
valid_idx = np.flatnonzero(valid)
valid_cells = pd.DataFrame(
    {
        "replicate": replicate.to_numpy()[valid],
        "group": group.astype(str).to_numpy()[valid],
    }
)
sample_key = valid_cells["replicate"] + "\x1f" + valid_cells["group"]
sample_codes, _ = pd.factorize(sample_key, sort=False)

In [31]:
incidence = sp.csr_matrix(
    (np.ones(len(valid_idx)), (sample_codes, valid_idx)),
    shape=(sample_codes.max() + 1, adata.n_obs),
)
pb_counts = dense_integer_counts(incidence @ counts)
pb_meta = valid_cells.groupby(sample_key, sort=False).agg(
    _pb_replicate=("replicate", "first"),
    _pb_group=("group", "first"),
    n_cells=("group", "size"),
)
pb_meta.index = [f"pb_{i}" for i in range(len(pb_meta))]

In [32]:
categories = [str(category) for category in group.cat.categories]
retained = [
    category
    for category in categories
    if pb_meta.loc[
        (pb_meta["_pb_group"] == category) & (pb_meta["n_cells"] >= MIN_CELLS),
        "_pb_replicate",
    ].nunique()
    >= MIN_REPLICATES
]
model_mask = pb_meta["_pb_group"].isin(retained) & (pb_meta["n_cells"] >= MIN_CELLS)
model_counts = pb_counts[model_mask.to_numpy()]
model_meta = pb_meta.loc[model_mask, ["_pb_replicate", "_pb_group"]].copy()
model_meta["_pb_replicate"] = pd.Categorical(model_meta["_pb_replicate"].astype(str))
model_meta["_pb_group"] = pd.Categorical(
    model_meta["_pb_group"].astype(str), categories=retained
)

In [33]:
gene_keep = model_counts.sum(axis=0) >= int(MIN_GENE_COUNTS)
model_counts = model_counts[:, gene_keep]
model_genes = adata.var_names.astype(str)[gene_keep]
model_pairs = set(
    zip(model_meta["_pb_replicate"].astype(str), model_meta["_pb_group"].astype(str))
)
model_cell_mask = valid & np.fromiter(
    (
        (str(r), str(g)) in model_pairs
        for r, g in zip(replicate.to_numpy(), group.astype(str).to_numpy())
    ),
    dtype=bool,
    count=adata.n_obs,
)
source_cell_mask = model_cell_mask & (group.astype(str).to_numpy() == str(SOURCE))
reference_cell_mask = model_cell_mask & (group.astype(str).to_numpy() == str(REFERENCE))

print(
    f"Pseudobulk model: {model_counts.shape[0]:,} samples x {model_counts.shape[1]:,} genes; retained categories: {len(retained):,}"
)

Pseudobulk model: 2,042 samples x 5,101 genes; retained categories: 16


In [34]:
dds = DeseqDataSet(
    counts=pd.DataFrame(model_counts, index=model_meta.index, columns=model_genes),
    metadata=model_meta,
    design="~ _pb_replicate + _pb_group",
    fit_type=DESEQ2_FIT_TYPE,
    n_cpus=N_CPUS,
    quiet=True,
)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    try:
        dds.deseq2(fit_type=DESEQ2_FIT_TYPE)
    except TypeError:
        dds.deseq2()

if "LFC" not in dds.varm:
    raise RuntimeError('DESeq2 fitting did not complete: dds.varm["LFC"] is missing.')

/Users/bastien.herve/miniconda3/envs/karospace-pseudobulk/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
if "LFC" not in dds.varm:
    raise RuntimeError(
        "dds is not fitted; rerun the previous cell containing dds.deseq2()."
    )

contrast = np.asarray(
    dds.contrast(column="_pb_group", baseline=REFERENCE, group_to_compare=SOURCE),
    dtype=float,
)
min_pct = (
    float(MIN_PCT_EXPRESSED) / 100
    if float(MIN_PCT_EXPRESSED) > 1
    else float(MIN_PCT_EXPRESSED)
)
gene_pos = {str(gene): i for i, gene in enumerate(adata.var_names.astype(str))}
model_cols = [gene_pos[gene] for gene in model_genes]
if min_pct > 0:
    source_pct = pct_positive(counts, source_cell_mask, model_cols)
    reference_pct = pct_positive(counts, reference_cell_mask, model_cols)
    test_genes = [
        gene
        for gene, spct, rpct in zip(model_genes, source_pct, reference_pct)
        if float(spct) >= min_pct or float(rpct) >= min_pct
    ]
else:
    test_genes = list(model_genes)
def subset_fitted_deseq2_dataset(dds_obj, selected_genes):
    selected_genes = [str(gene) for gene in selected_genes]
    if not selected_genes:
        return None
    if len(selected_genes) == int(dds_obj.n_vars) and list(map(str, dds_obj.var_names)) == selected_genes:
        return dds_obj
    subset = dds_obj[:, selected_genes].copy()
    try:
        if subset.__class__ is not dds_obj.__class__:
            subset.__class__ = dds_obj.__class__
    except Exception:
        pass
    for attr in (
        "refit_cooks",
        "low_memory",
        "formulaic_contrasts",
        "inference",
        "quiet",
        "fit_type",
        "min_replicates",
        "min_disp",
        "max_disp",
        "beta_tol",
        "min_mu",
        "max_iter",
        "n_cpus",
        "design_factors",
        "ref_level",
        "control_genes",
    ):
        if hasattr(dds_obj, attr):
            try:
                setattr(subset, attr, getattr(dds_obj, attr))
            except Exception:
                pass
    if "non_zero" in subset.var:
        non_zero = subset.var["non_zero"].to_numpy(dtype=bool)
    else:
        non_zero = np.ones(int(subset.n_vars), dtype=bool)
    subset.non_zero_idx = np.arange(int(subset.n_vars))[non_zero]
    subset.non_zero_genes = subset.var_names[non_zero]
    original_new_zeroes = getattr(dds_obj, "new_all_zeroes_genes", pd.Index([]))
    subset_names = {str(gene) for gene in subset.var_names}
    subset.new_all_zeroes_genes = pd.Index(
        [str(gene) for gene in original_new_zeroes if str(gene) in subset_names]
    )
    return subset

print(f"MIN_PCT_EXPRESSED retained {len(test_genes):,} of {len(model_genes):,} fitted genes for DeseqStats.")
if test_genes:
    stats_dds = subset_fitted_deseq2_dataset(dds, test_genes)
    stats = DeseqStats(stats_dds, contrast=contrast, quiet=True, n_cpus=N_CPUS)
    stats.summary()
else:
    stats = None


In [ ]:
raw = (
    stats.results_df.copy()
    if stats is not None
    else pd.DataFrame(columns=["baseMean", "log2FoldChange", "stat", "pvalue", "padj"])
)
raw["padj"] = padjust(raw["pvalue"], P_ADJUST_METHOD)

result_genes = raw.index.astype(str).to_list()
result_cols = [gene_pos[gene] for gene in result_genes]

de_table = pd.DataFrame(
    {
        "gene": result_genes,
        "baseMean": raw["baseMean"].to_numpy(float),
        "log2FoldChange": raw["log2FoldChange"].to_numpy(float),
        "stat": raw["stat"].to_numpy(float),
        "pvalue": raw["pvalue"].to_numpy(float),
        "padj": raw["padj"].to_numpy(float),
        "pct_source": pct_positive(counts, source_cell_mask, result_cols),
        "pct_reference": pct_positive(counts, reference_cell_mask, result_cols),
    }
)
de_table["max_pct"] = de_table[["pct_source", "pct_reference"]].max(axis=1)
de_table["is_de"] = (
    (de_table["padj"] < PADJ_CUTOFF)
    & (de_table["log2FoldChange"].abs() >= LOG2FC_CUTOFF)
)
de_table = de_table.sort_values(
    ["padj", "pvalue", "log2FoldChange"], ascending=[True, True, False]
).reset_index(drop=True)
de_table.head(20)


In [48]:
de_table["is_de"].value_counts()

is_de
False    4643
True      458
Name: count, dtype: int64

In [49]:
import gseapy as gp

if PATHWAY_GMT:
    gene_sets = read_gmt(PATHWAY_GMT)
    pathway_source = {"source": "gmt", "gene_set_count": len(gene_sets)}
else:
    libraries = gp.get_library_name(organism=PATHWAY_ORGANISM)
    reactome_libraries = [name for name in libraries if "reactome" in str(name).lower()]
    reactome_library = sorted(
        reactome_libraries,
        key=lambda name: (
            max(
                [
                    int(x)
                    for x in "".join(
                        ch if ch.isdigit() else " " for ch in str(name)
                    ).split()
                ]
                or [0]
            ),
            str(name),
        ),
        reverse=True,
    )[0]
    gene_sets = {
        term: clean_gene_set(genes)
        for term, genes in gp.get_library(
            name=reactome_library, organism=PATHWAY_ORGANISM
        ).items()
    }
    pathway_source = {
        "source": "reactome",
        "library": reactome_library,
        "organism": PATHWAY_ORGANISM,
        "gene_set_count": len(gene_sets),
    }

In [50]:
pathway_source

{'source': 'reactome',
 'library': 'Reactome_Pathways_2024',
 'organism': 'Mouse',
 'gene_set_count': 2100}

In [51]:
universe = de_table["gene"].astype(str).tolist()
up_genes = (
    de_table.loc[de_table["is_de"] & (de_table["log2FoldChange"] > 0), "gene"]
    .astype(str)
    .tolist()
)
down_genes = (
    de_table.loc[de_table["is_de"] & (de_table["log2FoldChange"] < 0), "gene"]
    .astype(str)
    .tolist()
)
ora_up = ora(up_genes, gene_sets, universe)
ora_down = ora(down_genes, gene_sets, universe)

In [53]:
fallback_score = np.sign(de_table["log2FoldChange"]) * -np.log10(
    np.clip(de_table["pvalue"], np.nextafter(0, 1), 1)
)
ranked = pd.DataFrame(
    {
        "gene": de_table["gene"].astype(str),
        "score": de_table["stat"].where(np.isfinite(de_table["stat"]), fallback_score),
    }
)
ranked = (
    ranked[np.isfinite(ranked["score"])]
    .drop_duplicates("gene")
    .sort_values("score", ascending=False)
)

In [ ]:
pre_res = gp.prerank(
    rnk=ranked,
    gene_sets=gene_sets,
    min_size=PATHWAY_MIN_OVERLAP,
    max_size=PATHWAY_MAX_SIZE,
    permutation_num=PATHWAY_GSEA_PERMUTATIONS,
    threads=N_CPUS,
    seed=PATHWAY_GSEA_SEED,
    outdir=None,
    no_plot=True,
    verbose=False,
)
gsea = pre_res.res2d.head(PATHWAY_TOP_N)

pathway_source, ora_up.head(), ora_down.head(), gsea.head()

In [ ]:
# Compact plots matching the HTML viewer behavior.
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize


def _empty_axis(ax, title, message="No pathways"):
    ax.axis("off")
    ax.set_title(title)
    ax.text(0.5, 0.5, message, ha="center", va="center", transform=ax.transAxes)


def _short_term(value, max_len=58):
    text = str(value or "")
    return text if len(text) <= max_len else text[: max_len - 1] + "…"


def plot_ora_dotplot(ax, table, title):
    if table is None or table.empty:
        _empty_axis(ax, title)
        return
    plot_df = table.sort_values(
        ["GeneRatio", "Adjusted P-value"], ascending=[False, True]
    ).head(12)
    plot_df = plot_df.sort_values("GeneRatio", ascending=True)
    score = plot_df["-log10 adjusted P-value"].to_numpy(float)
    count = plot_df["Count"].to_numpy(float)
    size_range = max(float(np.nanmax(count) - np.nanmin(count)), 1.0)
    sizes = 55 + np.sqrt(np.maximum(count - np.nanmin(count), 0) / size_range) * 260
    norm = Normalize(vmin=float(np.nanmin(score)), vmax=float(np.nanmax(score)) + 1e-12)
    y = np.arange(len(plot_df))
    scatter = ax.scatter(
        plot_df["GeneRatio"],
        y,
        s=sizes,
        c=score,
        cmap="viridis",
        norm=norm,
        edgecolors="none",
        alpha=0.9,
    )
    ax.set_yticks(y)
    ax.set_yticklabels([_short_term(term) for term in plot_df["Term"]], fontsize=8)
    ax.set_xlabel("GeneRatio")
    ax.set_title(title)
    ax.grid(axis="x", color="0.9", linewidth=0.8)
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("-log10 adjusted p")


def _gsea_result(term):
    return (getattr(pre_res, "results", {}) or {}).get(term, {})


def _series_values(value):
    if hasattr(value, "to_numpy"):
        return value.to_numpy(dtype=float)
    return np.asarray(value, dtype=float)


def _scale_to_band(values, low, high):
    arr = np.asarray(values, dtype=float)
    finite = np.concatenate([arr[np.isfinite(arr)], np.array([0.0])])
    if finite.size == 0 or np.isclose(np.nanmin(finite), np.nanmax(finite)):
        return np.full_like(arr, (low + high) / 2, dtype=float)
    return low + (arr - np.nanmin(finite)) / (np.nanmax(finite) - np.nanmin(finite)) * (
        high - low
    )


def plot_gsea_term(ax, term):
    result = _gsea_result(term)
    res = np.asarray(result.get("RES", []), dtype=float)
    hits = np.asarray(result.get("hits", []), dtype=float)
    rank_metric = _series_values(getattr(pre_res, "ranking", []))
    if res.size == 0 or hits.size == 0 or rank_metric.size == 0:
        _empty_axis(ax, "GSEA enrichment", "No running profile data")
        return
    nes = float(result.get("nes", np.nan))
    es = float(result.get("es", np.nan))
    fdr = float(result.get("fdr", np.nan))
    display_sign = -1.0 if np.isfinite(nes) and nes < 0 else 1.0
    profile_color = "#d94f4f" if display_sign > 0 else "#4f82d9"
    opposite_color = "#4f82d9" if display_sign > 0 else "#d94f4f"
    label_a = SOURCE if display_sign > 0 else REFERENCE
    label_b = REFERENCE if display_sign > 0 else SOURCE
    rank_count = len(res)
    x = np.arange(rank_count)

    ax.set_title(f"GSEA enrichment: {_short_term(term, 72)}", fontsize=10)
    ax.set_xlim(0, rank_count - 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel("Rank in ordered dataset")
    for spine in ["left", "right", "top"]:
        ax.spines[spine].set_visible(False)

    es_low, es_high = 0.63, 0.98
    hit_low, hit_high = 0.45, 0.58
    metric_low, metric_high = 0.08, 0.38
    for low, high in [
        (es_low, es_high),
        (hit_low, hit_high),
        (metric_low, metric_high),
    ]:
        ax.axhspan(low, high, color="none", ec="0.82", lw=0.8)
        for frac in [0.25, 0.5, 0.75]:
            ax.axhline(
                low + (high - low) * frac, color="0.9", lw=0.7, ls="--", zorder=0
            )

    es_plot = display_sign * res
    es_y = _scale_to_band(es_plot, es_low, es_high)
    ax.axhline(
        _scale_to_band(np.array([0.0]), es_low, es_high)[0], color="0.65", lw=0.8
    )
    ax.plot(x, es_y, color=profile_color, lw=2.2)
    ax.vlines(hits, hit_low + 0.01, hit_high - 0.01, color="black", lw=0.8)

    cmap = LinearSegmentedColormap.from_list(
        "gsea_direction", [profile_color, "#f5f5f5", opposite_color]
    )
    ax.imshow(
        np.linspace(0, 1, 256).reshape(1, -1),
        extent=[0, rank_count - 1, hit_low - 0.035, hit_low - 0.005],
        aspect="auto",
        cmap=cmap,
        interpolation="nearest",
    )

    metric = display_sign * rank_metric[:rank_count]
    metric_y = _scale_to_band(metric, metric_low, metric_high)
    zero_metric = _scale_to_band(np.array([0.0]), metric_low, metric_high)[0]
    ax.axhline(zero_metric, color="0.65", lw=0.8)
    ax.vlines(x, zero_metric, metric_y, color="0.72", lw=1.0, alpha=0.9)
    ax.text(
        0.01,
        hit_low - 0.055,
        f"{label_a} enriched",
        color=profile_color,
        ha="left",
        va="top",
        transform=ax.transAxes,
        fontsize=8,
    )
    ax.text(
        0.99,
        hit_low - 0.055,
        f"{label_b} enriched",
        color=opposite_color,
        ha="right",
        va="top",
        transform=ax.transAxes,
        fontsize=8,
    )
    ax.text(
        0.01,
        0.01,
        f"NES={abs(nes):.3g}; ES={abs(es):.3g}; adj. p={fdr:.3g}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8,
        color="0.35",
    )


fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
plot_ora_dotplot(axes[0], ora_up, f"ORA pathways: {SOURCE}")
plot_ora_dotplot(axes[1], ora_down, f"ORA pathways: {REFERENCE}")
plt.show()

if gsea.empty:
    print("No GSEA pathways available.")
else:
    terms = gsea["Term"].astype(str).tolist()

    def _plot_selected_gsea(index):
        fig, ax = plt.subplots(1, 1, figsize=(12, 6), constrained_layout=True)
        plot_gsea_term(ax, terms[int(index)])
        plt.show()

    options = []
    for idx, row in gsea.iterrows():
        nes = abs(float(row.get("NES", np.nan)))
        fdr = float(row.get("FDR q-val", np.nan)) if "FDR q-val" in row else np.nan
        options.append(
            (
                f"{_short_term(row['Term'], 72)} | NES={nes:.3g}, adj. p={fdr:.3g}",
                len(options),
            )
        )
    try:
        import ipywidgets as widgets
        from IPython.display import display

        selector = widgets.Dropdown(
            options=options, description="Pathway:", layout=widgets.Layout(width="100%")
        )
        output = widgets.interactive_output(_plot_selected_gsea, {"index": selector})
        display(widgets.VBox([selector, output]))
    except Exception as exc:
        print(
            "ipywidgets is unavailable; showing the first retained GSEA pathway instead."
        )
        print(exc)
        _plot_selected_gsea(0)